In [ ]:
# ============================================================
# 0. MOUNT DRIVE AND INSTALL RASTERIO
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install rasterio pyproj -q

import os, re, glob
import numpy as np
import pandas as pd
import rasterio
from rasterio.merge import merge as rio_merge
from rasterio.windows import from_bounds
from pyproj import Transformer
import time, warnings
warnings.filterwarnings('ignore')

print("Ready.")

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

DRIVE_ROOT = '/content/drive/MyDrive/Thesis/2025-Data'
POINTS_CSV = f'{DRIVE_ROOT}/prediction_points.csv'

# All images live in a flat folder (no subfolders).
# Supports both single-file and multi-part GEE exports:
#   Single:  Zambales_2025_Q1.tif
#   Split:   output_Zambales_2025_Q1-0000000000-0000000000.tif
#            output_Zambales_2025_Q1-0000014848-0000000000.tif
IMAGE_ROOT       = f'{DRIVE_ROOT}/Sentinel2'
TILE_OUTPUT      = '/content/tiles'
DRIVE_OUTPUT_DIR = f'{DRIVE_ROOT}/tiles_zipped'
CHECKPOINT       = f'{DRIVE_ROOT}/tile_checkpoint.csv'

os.makedirs(TILE_OUTPUT,      exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

TILE_HALF_SIZE_M = 5000   # 10 km x 10 km tiles
QUARTERS = ['Q1', 'Q2', 'Q3', 'Q4']

# Maps Province Name -> file prefix used in GEE export filename
PROVINCE_PREFIX_MAP = {
    'Ilocos Norte'       : 'Ilocos_Norte',
    'Pampanga'           : 'Pampanga',
    'Benguet'            : 'Benguet',
    'Kalinga'            : 'Kalinga',
    'Aklan'              : 'Aklan',
    'Zamboanga del Norte': 'Zamboanga_del_Norte',
    'Basilan'            : 'Basilan',
    'Tawi-Tawi'          : 'Tawi-Tawi',
    'Maguindanao del Sur': 'Maguindanao_del_Sur',
    'Davao Oriental'     : 'Davao_Oriental',
    'NCR'                : 'NCR',
    # New provinces
    'Bataan'             : 'Bataan',
    'Zambales'           : 'Zambales',
    'Occidental Mindoro' : 'Occidental_Mindoro',
    'Oriental Mindoro'   : 'Oriental_Mindoro',
}

print(f"Configured {len(PROVINCE_PREFIX_MAP)} provinces.")

In [ ]:
# ============================================================
# 2. FILE DISCOVERY — handles single and split GEE exports
# ============================================================

def find_quarter_files(image_root, prefix, quarter):
    """
    Return sorted list of file paths for a province+quarter.
    Handles three naming patterns from GEE:
      1. {prefix}_2025_{quarter}.tif
      2. output_{prefix}_2025_{quarter}.tif
      3. output_{prefix}_2025_{quarter}-NNNNNN-NNNNNN.tif  (split)
    Returns [] if nothing is found.
    """
    base = f"{prefix}_2025_{quarter}"

    # Pattern 1: plain single file
    p1 = os.path.join(image_root, f"{base}.tif")
    if os.path.exists(p1):
        return [p1]

    # Pattern 2: single file with 'output_' prefix
    p2 = os.path.join(image_root, f"output_{base}.tif")
    if os.path.exists(p2):
        return [p2]

    # Pattern 3: split parts (with or without 'output_' prefix)
    parts = sorted(glob.glob(os.path.join(image_root, f"{base}-*.tif"))) or \
            sorted(glob.glob(os.path.join(image_root, f"output_{base}-*.tif")))
    return parts


# Verify all files
print("Checking image files...\n")
ok_count, missing = 0, []

for prov, prefix in PROVINCE_PREFIX_MAP.items():
    for q in QUARTERS:
        files = find_quarter_files(IMAGE_ROOT, prefix, q)
        if files:
            total_mb  = sum(os.path.getsize(f) / 1e6 for f in files)
            parts_str = f" ({len(files)} parts)" if len(files) > 1 else ""
            print(f"  OK  : {prefix}_2025_{q}{parts_str}  {total_mb:.0f} MB")
            ok_count += 1
        else:
            print(f"  MISS: {prefix}_2025_{q}")
            missing.append((prov, q))

print(f"\nFound: {ok_count}/{len(PROVINCE_PREFIX_MAP)*4}, Missing: {len(missing)}")
if missing:
    for prov, q in missing:
        print(f"  Missing: {prov} {q}")

In [ ]:
# ============================================================
# 3. LOAD POINTS AND CHECKPOINT
# ============================================================

df_points = pd.read_csv(POINTS_CSV)
print(f"Total prediction points: {len(df_points)}")
print("\nPoints per province:")
print(df_points['Province'].value_counts().to_string())

if os.path.exists(CHECKPOINT):
    done_ids = set(pd.read_csv(CHECKPOINT)['PointID'].tolist())
    print(f"\nCheckpoint: {len(done_ids)} points already tiled.")
else:
    done_ids = set()
    print("\nNo checkpoint. Starting fresh.")

remaining = df_points[~df_points['PointID'].isin(done_ids)]
print(f"Remaining: {len(remaining)} points")

In [ ]:
# ============================================================
# 4. TILING FUNCTION — supports split multi-part source files
# ============================================================

to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:32651', always_xy=True)


def crop_tile(src_paths, center_lon, center_lat, half_size_m, out_path):
    """
    Crop a tile from one or more province GeoTIFF parts.

    src_paths   : list of file paths (1 for single-file, 2+ for split exports)
    center_lon  : WGS84 longitude of tile centre
    center_lat  : WGS84 latitude of tile centre
    half_size_m : half tile edge in metres (5000 -> 10 km tile)
    out_path    : output file path

    For single-file provinces the function behaves identically to the
    original implementation.  For split provinces, rasterio.merge is used
    to stitch the overlapping parts into one tile in memory before saving.

    Returns True on success, False if no source part overlaps the tile area.
    """
    cx, cy   = to_utm.transform(center_lon, center_lat)
    minx_utm = cx - half_size_m
    maxx_utm = cx + half_size_m
    miny_utm = cy - half_size_m
    maxy_utm = cy + half_size_m

    # Open only the parts that spatially overlap the tile bounding box
    overlapping = []
    for path in src_paths:
        ds = rasterio.open(path)
        if str(ds.crs) != 'EPSG:32651':
            ds.close()
            raise ValueError(
                f"Image {path} is not EPSG:32651 (got {ds.crs}). "
                "Re-export from GEE with crs='EPSG:32651'."
            )
        b = ds.bounds
        if minx_utm < b.right and maxx_utm > b.left and \
                miny_utm < b.top and maxy_utm > b.bottom:
            overlapping.append(ds)
        else:
            ds.close()

    if not overlapping:
        return False

    try:
        if len(overlapping) == 1:
            # ── Single part: original window-crop approach ────────────────
            src  = overlapping[0]
            b    = src.bounds
            minx_s = max(minx_utm, b.left)
            miny_s = max(miny_utm, b.bottom)
            maxx_s = min(maxx_utm, b.right)
            maxy_s = min(maxy_utm, b.top)

            if minx_s >= maxx_s or miny_s >= maxy_s:
                return False

            window  = from_bounds(minx_s, miny_s, maxx_s, maxy_s,
                                  transform=src.transform)
            data    = src.read(window=window)
            profile = src.profile.copy()
            profile.update({
                'height'   : data.shape[1],
                'width'    : data.shape[2],
                'transform': src.window_transform(window),
            })

        else:
            # ── Multiple parts: merge then crop ───────────────────────────
            merged_data, merged_transform = rio_merge(
                overlapping,
                bounds=(minx_utm, miny_utm, maxx_utm, maxy_utm),
                nodata=overlapping[0].nodata,
            )
            data    = merged_data
            profile = overlapping[0].profile.copy()
            profile.update({
                'height'   : data.shape[1],
                'width'    : data.shape[2],
                'transform': merged_transform,
            })

        if data.shape[1] == 0 or data.shape[2] == 0:
            return False

        profile.update({'driver': 'GTiff', 'compress': 'lzw'})
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data)
        return True

    finally:
        for ds in overlapping:
            ds.close()

In [ ]:
# ============================================================
# 5A. SINGLE PROVINCE (manual, for testing or resuming one area)
# ============================================================

CURRENT_PROVINCE = 'Zambales'   # <- change or set to None to skip

if CURRENT_PROVINCE and CURRENT_PROVINCE in PROVINCE_PREFIX_MAP:
    prefix   = PROVINCE_PREFIX_MAP[CURRENT_PROVINCE]
    prov_pts = remaining[remaining['Province'] == CURRENT_PROVINCE]
    print(f"Processing: {CURRENT_PROVINCE} ({len(prov_pts)} points)")

    src_paths = {}
    for q in QUARTERS:
        files = find_quarter_files(IMAGE_ROOT, prefix, q)
        src_paths[q] = files
        if len(files) > 1:
            print(f"  INFO: {q} has {len(files)} split parts")
        elif len(files) == 0:
            print(f"  WARNING: no files for {q}")

    completed, failed_list = [], []
    t_start = time.time()

    for i, (_, row) in enumerate(prov_pts.iterrows()):
        pid     = int(row['PointID'])
        pid_str = str(pid).zfill(5)
        out_dir = os.path.join(TILE_OUTPUT, pid_str)
        os.makedirs(out_dir, exist_ok=True)

        for q in QUARTERS:
            out_path = os.path.join(out_dir, f'point_{pid_str}_2025_{q}.tif')
            if os.path.exists(out_path):
                continue
            if not src_paths[q]:
                failed_list.append({'PointID': pid, 'Quarter': q, 'Error': 'no_file'})
                continue
            try:
                ok = crop_tile(src_paths[q], row['Longitude'],
                               row['Latitude'], TILE_HALF_SIZE_M, out_path)
                if not ok:
                    failed_list.append({'PointID': pid, 'Quarter': q,
                                        'Error': 'no_overlap'})
            except Exception as e:
                failed_list.append({'PointID': pid, 'Quarter': q,
                                    'Error': str(e)[:120]})

        completed.append({'PointID': pid, 'Province': CURRENT_PROVINCE})

        if (i + 1) % 50 == 0 or (i + 1) == len(prov_pts):
            elapsed = time.time() - t_start
            rate    = (i + 1) / elapsed * 60 if elapsed > 0 else 0
            print(f"  {i+1}/{len(prov_pts)} ({rate:.0f} pts/min)  "
                  f"failed: {len(failed_list)}")

    # Checkpoint
    df_new = pd.DataFrame(completed)
    if os.path.exists(CHECKPOINT):
        df_new = pd.concat([pd.read_csv(CHECKPOINT), df_new], ignore_index=True)
    df_new.to_csv(CHECKPOINT, index=False)

    print(f"\nDone: {len(completed)} pts, {len(failed_list)} failed, "
          f"{time.time()-t_start:.0f}s")
    if failed_list:
        print(pd.DataFrame(failed_list)['Error'].value_counts().to_string())

elif CURRENT_PROVINCE:
    print(f"'{CURRENT_PROVINCE}' not in PROVINCE_PREFIX_MAP.")

In [ ]:
# ============================================================
# 5B. AUTO-PROCESS ALL REMAINING PROVINCES
# ============================================================
import shutil

if os.path.exists(CHECKPOINT):
    done_ids_reload = set(pd.read_csv(CHECKPOINT)['PointID'].tolist())
else:
    done_ids_reload = set()

remaining_reload = df_points[~df_points['PointID'].isin(done_ids_reload)]
remaining_provs  = remaining_reload['Province'].unique()
print(f"Provinces remaining: {len(remaining_provs)}")

for prov_name in remaining_provs:
    if prov_name not in PROVINCE_PREFIX_MAP:
        print(f"\nSKIP: '{prov_name}' not in PROVINCE_PREFIX_MAP.")
        continue

    prefix   = PROVINCE_PREFIX_MAP[prov_name]
    prov_pts = remaining_reload[remaining_reload['Province'] == prov_name]
    print(f"\n{'='*55}")
    print(f"Processing: {prov_name} ({len(prov_pts)} points)")
    print(f"{'='*55}")

    # Resolve files (handles single + split)
    src_paths = {}
    for q in QUARTERS:
        files = find_quarter_files(IMAGE_ROOT, prefix, q)
        src_paths[q] = files
        if len(files) > 1:
            print(f"  INFO: {q} -> {len(files)} split parts")
        elif len(files) == 0:
            print(f"  WARNING: {q} -> no files found")

    completed, failed_list = [], []
    t_start = time.time()

    for i, (_, row) in enumerate(prov_pts.iterrows()):
        pid     = int(row['PointID'])
        pid_str = str(pid).zfill(5)
        out_dir = os.path.join(TILE_OUTPUT, pid_str)
        os.makedirs(out_dir, exist_ok=True)

        for q in QUARTERS:
            out_path = os.path.join(out_dir, f'point_{pid_str}_2025_{q}.tif')
            if os.path.exists(out_path):
                continue
            if not src_paths[q]:
                failed_list.append({'PointID': pid, 'Quarter': q,
                                    'Error': 'no_file'})
                continue
            try:
                ok = crop_tile(src_paths[q], row['Longitude'],
                               row['Latitude'], TILE_HALF_SIZE_M, out_path)
                if not ok:
                    failed_list.append({'PointID': pid, 'Quarter': q,
                                        'Error': 'no_overlap'})
            except Exception as e:
                failed_list.append({'PointID': pid, 'Quarter': q,
                                    'Error': str(e)[:120]})

        completed.append({'PointID': pid, 'Province': prov_name})

        if (i + 1) % 50 == 0 or (i + 1) == len(prov_pts):
            elapsed = time.time() - t_start
            rate    = (i + 1) / elapsed * 60 if elapsed > 0 else 0
            print(f"  {i+1}/{len(prov_pts)} ({rate:.0f} pts/min)  "
                  f"failed: {len(failed_list)}")

    # Zip to Drive
    print(f"  Packaging {prov_name}...")
    zip_name = f"tiles_{prefix}"
    shutil.make_archive(f"/content/{zip_name}", 'zip', TILE_OUTPUT)
    drive_target = os.path.join(DRIVE_OUTPUT_DIR, f"{zip_name}.zip")
    shutil.move(f"/content/{zip_name}.zip", drive_target)
    print(f"  Saved: {drive_target}")

    shutil.rmtree(TILE_OUTPUT)
    os.makedirs(TILE_OUTPUT, exist_ok=True)

    # Checkpoint
    df_new = pd.DataFrame(completed)
    if os.path.exists(CHECKPOINT):
        df_new = pd.concat([pd.read_csv(CHECKPOINT), df_new], ignore_index=True)
    df_new.to_csv(CHECKPOINT, index=False)

    print(f"  Done: {len(completed)} pts, {len(failed_list)} failed, "
          f"{time.time()-t_start:.0f}s")

print("\nAll provinces processed.")

In [ ]:
# ============================================================
# 6. VERIFY OUTPUT
# ============================================================

tile_counts = []
for pid_str in os.listdir(TILE_OUTPUT):
    d = os.path.join(TILE_OUTPUT, pid_str)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.endswith('.tif')])
        tile_counts.append({'PointID': pid_str, 'n_tiles': n})

df_counts = pd.DataFrame(tile_counts)
if df_counts.empty:
    print("No tiles in local folder (already zipped to Drive).")
else:
    print(f"Total folders : {len(df_counts)}")
    print(f"Complete (4/4): {(df_counts['n_tiles'] == 4).sum()}")
    print(f"Incomplete    : {(df_counts['n_tiles'] < 4).sum()}")
    print(f"Empty (0)     : {(df_counts['n_tiles'] == 0).sum()}")
    print(f"Total tiles   : {df_counts['n_tiles'].sum()}")
    if (df_counts['n_tiles'] < 4).any():
        print("\nIncomplete:")
        print(df_counts[df_counts['n_tiles'] < 4].to_string(index=False))

In [ ]:
# ============================================================
# 7. VISUAL SPOT CHECK
# ============================================================
import matplotlib.pyplot as plt

complete_dirs = df_counts[df_counts['n_tiles'] == 4]['PointID'].tolist()
samples = np.random.choice(complete_dirs, min(3, len(complete_dirs)), replace=False)

fig, axes = plt.subplots(len(samples), 4, figsize=(16, 4 * len(samples)))
if len(samples) == 1:
    axes = axes.reshape(1, -1)

for row_i, pid_str in enumerate(samples):
    d    = os.path.join(TILE_OUTPUT, pid_str)
    tifs = sorted([f for f in os.listdir(d) if f.endswith('.tif')])
    for col_i, tif in enumerate(tifs[:4]):
        ax = axes[row_i, col_i]
        with rasterio.open(os.path.join(d, tif)) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0)).astype(float)
            p2, p98 = np.percentile(img, (2, 98))
            if p98 > p2:
                img = np.clip((img - p2) / (p98 - p2), 0, 1)
            ax.imshow(img)
            ax.set_title(tif.split('_')[-1].replace('.tif',''), fontsize=10)
            ax.axis('off')
    axes[row_i, 0].set_ylabel(f'Point {pid_str}', fontsize=11,
                               rotation=0, labelpad=60, va='center')

plt.suptitle('Tile spot check', fontsize=13)
plt.tight_layout()
plt.show()